<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Selecting_Embedding_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Selecting the Right Embedding Model

The embedder decides what your retriever can *find*. Swap it and every score, every ranking, and every eval number in your pipeline moves — which is why "just use the default" is a decision, not the absence of one.

This notebook makes the decision measurable. Three contenders — the course default API model, a retrieval specialist (Cohere `embed-v4.0`), and an open-weight model running on your own machine — index the *same* chunks, answer the *same* questions, and land in one table.

📎 *`embed()` is the course helper from Basic RAG; `embed_cohere()` and `embed_local()` are new — all three defined below in plain Python, alongside the same ingestion and evaluation functions you built in Sections 3–4. No framework, no course package.*

## 🧭 What You'll Learn

- The five criteria that actually decide an embedding model — and why the leaderboard only shortlists
- **Asymmetry**: why documents and queries must be embedded differently, in three dialects (Gemini `task_type`, Cohere `input_type`, open-model query instructions)
- Building three parallel indexes over identical chunks with one `embed_fn=` seam
- Measuring embedders on *your* corpus with hit rate and MRR — and why raw similarity scores never compare across models
- What it costs to change your mind later (spoiler: a full re-ingest)

## 1. Setup: Environment, Keys, and Providers

Three contenders means three credentials paths: your course provider key, a Cohere key ([free trial keys](https://dashboard.cohere.com/api-keys) are rate-limited but sufficient here), and — for the open model — no key at all, because it runs locally.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Pick your model providers (dropdowns in Colab; edit the values locally)
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]
EMBED_PROVIDER = "gemini"  # @param ["gemini", "openai"]  (Anthropic has no embedding API)

# Pick a model for the selected provider — or TYPE any newer model ID into the
# box (the dropdown is editable thanks to allow-input):
CHAT_MODEL = "gemini-3.7-flash"  # @param ["gemini-3.7-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}
EMBED_MODEL = "gemini-embedding-001"  # @param ["gemini-embedding-001", "text-embedding-3-small"] {allow-input: true}

# The open-weight contender — any sentence-transformers model ID works
OPEN_EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"  # @param ["Qwen/Qwen3-Embedding-0.6B", "Qwen/Qwen3-Embedding-8B", "BAAI/bge-small-en-v1.5", "intfloat/multilingual-e5-base"] {allow-input: true}

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = sorted({_KEY_FOR[PROVIDER], _KEY_FOR[EMBED_PROVIDER]}) + ["COHERE_API_KEY"]

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026). This lesson's two
    # extras: cohere (contender 2) and sentence-transformers (contender 3).
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
            "chromadb==1.5.9",
            "tiktoken==0.13.0",
            "cohere==7.0.8",
            "sentence-transformers==5.5.1",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon) → Add new secret → e.g. COHERE_API_KEY
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: dependencies are installed once from the repo's requirements.
    # Keys live in a .env file at the repo root.
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | chat: {PROVIDER} | embeddings: {EMBED_PROVIDER} | open contender: {OPEN_EMBED_MODEL}")

✅ Setup complete — local | chat: gemini | embeddings: gemini | open contender: Qwen/Qwen3-Embedding-0.6B


## 2. The Pipeline, Redefined in Plain Python

Three embedders, one ingestion function, one search function, one evaluator — all defined in the cells below, in full view. The whole comparison rests on the last three being *shared*: identical everywhere except the vectors.

In [2]:
# 📎 generate() and embed() — the two course helpers, exactly as built in
#    "How To Use LLMs via API" and "Basic RAG". embed() is contender 1.
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (August 2026)
MODELS = {
    "gemini": "gemini-3.7-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default:
MODELS[PROVIDER] = CHAT_MODEL
EMBED_MODELS = {
    "gemini": "gemini-embedding-001",
    "openai": "text-embedding-3-small",
}
EMBED_MODELS[EMBED_PROVIDER] = EMBED_MODEL  # setup-cell selection (or typed ID) wins
EMBED_DIM = 1536  # same output size for both providers, so the rest of the code never cares

# Create only the clients we actually need
if "gemini" in (PROVIDER, EMBED_PROVIDER):
    gemini_client = genai.Client()
if "openai" in (PROVIDER, EMBED_PROVIDER):
    openai_client = OpenAI()
if PROVIDER == "anthropic":
    anthropic_client = Anthropic()


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def embed(texts, task="document"):
    """Embed a list of texts with the selected EMBED_PROVIDER.

    Returns a list of EMBED_DIM-dimensional vectors (one per input text).
    task: "document" for corpus chunks, "query" for user questions —
    Gemini embeds the two slightly differently to improve retrieval.
    """
    if isinstance(texts, str):
        texts = [texts]
    # Embedding models work best on single-line inputs
    texts = [t.replace("\n", " ") for t in texts]

    if EMBED_PROVIDER == "gemini":
        result = gemini_client.models.embed_content(
            model=EMBED_MODELS["gemini"],
            contents=texts,
            config=genai_types.EmbedContentConfig(
                task_type="RETRIEVAL_DOCUMENT" if task == "document" else "RETRIEVAL_QUERY",
                output_dimensionality=EMBED_DIM,
            ),
        )
        return [e.values for e in result.embeddings]

    if EMBED_PROVIDER == "openai":
        result = openai_client.embeddings.create(
            model=EMBED_MODELS["openai"], input=texts
        )
        return [d.embedding for d in result.data]

    raise ValueError(f"Unknown EMBED_PROVIDER: {EMBED_PROVIDER!r}")

In [3]:
# The two NEW embedders — each a direct SDK call, each handling its own
# asymmetry dialect (Section 5 explains why that matters).
import cohere

cohere_client = cohere.ClientV2()  # reads COHERE_API_KEY from the environment


def embed_cohere(texts, task="document", model="embed-v4.0", output_dimension=1536, batch_size=96):
    """Cohere embeddings via ClientV2.embed — input_type set from `task`.

    embed-v4.0 is Matryoshka-trained (256/512/1024/1536): the dimension is a
    CHOICE, and the course pins 1536 to keep vectors comparable across lessons.
    """
    if isinstance(texts, str):
        texts = [texts]
    input_type = "search_document" if task == "document" else "search_query"
    vectors = []
    for start in range(0, len(texts), batch_size):  # the API caps texts per call
        result = cohere_client.embed(
            model=model,
            texts=texts[start : start + batch_size],
            input_type=input_type,
            output_dimension=output_dimension,
            embedding_types=["float"],
        )
        vectors.extend(result.embeddings.float_)
    return vectors


_ST_MODELS = {}  # sentence-transformers model cache — load once, reuse

# Query-side instructions for instruction-tuned retrieval families (July 2026).
# Documents embed plain; ONLY queries carry the instruction. The e5 family is
# different again: it prefixes BOTH sides ("query: " / "passage: ").
_QUERY_PROMPTS = {
    "qwen3-embedding": (
        "Instruct: Given a web search query, retrieve relevant passages that "
        "answer the query\nQuery: "
    ),
    "gte-qwen": (
        "Instruct: Given a web search query, retrieve relevant passages that "
        "answer the query\nQuery: "
    ),
}


def embed_local(texts, model_name, task="document", query_prompt=None):
    """Embed with a local sentence-transformers model.

    Applies the model family's query instruction (or your `query_prompt`) on
    task="query" only — except the e5 family, which wants a prefix on both sides.
    """
    from sentence_transformers import SentenceTransformer

    if isinstance(texts, str):
        texts = [texts]
    if model_name not in _ST_MODELS:
        _ST_MODELS[model_name] = SentenceTransformer(model_name)
    model = _ST_MODELS[model_name]
    name = model_name.lower()

    prefix = ""
    if "e5" in name:  # e5: prefix BOTH sides
        prefix = "query: " if task == "query" else "passage: "
    elif task == "query":  # instruction-tuned families: instruction on queries only
        if query_prompt is not None:
            prefix = query_prompt
        else:
            prefix = next((p for key, p in _QUERY_PROMPTS.items() if key in name), "")

    vectors = model.encode([prefix + t for t in texts], normalize_embeddings=True)
    return [v.tolist() for v in vectors]

In [4]:
# 📎 The pipeline functions from Sections 3–4 — chunk(), ingest(), search() —
#    with ONE upgrade for this lesson: an embed_fn= seam, so the same twenty
#    lines can index and query with ANY of the three embedders.
import chromadb
import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")


def n_tokens(text):
    """Number of tokens in a string."""
    return len(ENC.encode(text))


def chunk(text, chunk_size=512, chunk_overlap=128):
    """Split text into token-based chunks; consecutive chunks share chunk_overlap tokens."""
    tokens = ENC.encode(text)
    chunks = []
    step = chunk_size - chunk_overlap
    for start in range(0, len(tokens), step):
        window = tokens[start : start + chunk_size]
        chunks.append(ENC.decode(window))
        if start + chunk_size >= len(tokens):
            break  # the final window reached the end of the text
    return chunks


def _stored_dim(collection):
    """The vector width this collection is locked to, or None if it was never written."""
    dim = getattr(getattr(collection, "_model", None), "dimension", None)
    if dim is not None:
        return dim
    if collection.count():  # fallback for Chroma builds that don't expose it
        return len(collection.get(limit=1, include=["embeddings"])["embeddings"][0])
    return None


def get_collection(name, path="./embedding-bakeoff-chroma", embed_fn=None, reset=False):
    """A persistent Chroma collection with cosine distance.

    Chroma locks a collection's vector width on its FIRST write and keeps it for
    the life of the collection. Point a differently-sized embedder at that same
    name later — say a 3072-dim index from an earlier run, then this notebook's
    1536-dim setting — and the upsert fails with:

        InvalidArgumentError: Collection expecting embedding with dimension of 3072, got 1536

    Pass embed_fn= and this compares the locked width against what that embedder
    actually produces, rebuilding the collection when they disagree. Re-embedding
    from scratch IS the correct repair, not a workaround (Section 9): vectors of
    different widths live in unrelated spaces and must never share an index.
    Pass reset=True to force a clean rebuild regardless.
    """
    client = chromadb.PersistentClient(path=path)

    def _open():
        return client.get_or_create_collection(name=name, metadata={"hnsw:space": "cosine"})

    if reset:
        try:
            client.delete_collection(name)
        except chromadb.errors.NotFoundError:
            pass  # nothing to reset on a first run
        return _open()

    collection = _open()
    if embed_fn is None:
        return collection

    stored = _stored_dim(collection)
    if stored is None:
        return collection  # never written, so there is nothing to clash with

    # One tiny probe call, and only for a collection that already exists.
    # NOTE: count() == 0 does NOT mean "safe" — deleting every record leaves the
    # width locked, so we ask the collection itself, not its contents.
    wanted = len(embed_fn("dimension probe", task="document")[0])
    if stored != wanted:
        print(f"↻ {name}: index is locked to {stored} dims, this embedder produces {wanted} — rebuilding")
        client.delete_collection(name)
        collection = _open()
    return collection


def ingest(articles, collection, chunk_size=512, chunk_overlap=128, batch_size=50, embed_fn=None):
    """chunk → embed (via embed_fn) → upsert with stable {title[:40]}-{i} ids. Idempotent."""
    embed_fn = embed_fn or embed
    records = []
    for art in articles:
        for i, piece in enumerate(chunk(art["text"], chunk_size, chunk_overlap)):
            records.append({
                "id": f"{art['title'][:40]}-{i}",
                "text": piece,
                "title": art["title"],
                "url": art["url"],
                "source": art["source"],
            })
    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]
        collection.upsert(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            embeddings=embed_fn([r["text"] for r in batch], task="document"),
            metadatas=[{"title": r["title"], "url": r["url"], "source": r["source"]} for r in batch],
        )
    return len(records)


def search(query, collection, top_k=5, embed_fn=None):
    """Dense retrieval against one collection, embedding the query with embed_fn."""
    embed_fn = embed_fn or embed
    result = collection.query(
        query_embeddings=[embed_fn(query, task="query")[0]], n_results=top_k
    )
    return [
        {"id": cid, "text": doc, "score": 1 - dist, **meta}
        for cid, doc, dist, meta in zip(
            result["ids"][0], result["documents"][0],
            result["distances"][0], result["metadatas"][0],
        )
    ]


def show_chunks(hits, max_chars=120):
    """Print retrieved chunks with scores — look before you trust."""
    for r in hits:
        preview = r["text"][:max_chars].replace("\n", " ")
        print(f"  {r['score']:.4f} | {r['title'][:40]:40} | {preview}…")

## 3. The Corpus: One Set of Chunks for Everybody

A fair embedding comparison needs **identical chunks** for every model — only the vectors may differ. `ingest()` derives chunk ids deterministically from each document, so the same corpus chunked with the same settings produces the same ids in all three indexes. That is what lets one eval set grade all three.

In [5]:
import csv
import pathlib

import requests

# Course dataset — hosted in the Towards AI org dataset repo on Hugging Face
DATA_URL = "https://huggingface.co/datasets/towardsai-tutors/full-stack-ai-engineering-data/resolve/main/ai-docs.csv"
DATA_PATH = pathlib.Path("ai-docs.csv")

if not DATA_PATH.exists():
    DATA_PATH.write_bytes(requests.get(DATA_URL, timeout=30).content)
    print("Downloaded", DATA_PATH)

# Read the articles WITH their metadata (title, url, source) — we carry it everywhere
articles = []
with open(DATA_PATH, mode="r", encoding="utf-8") as f:
    for idx, row in enumerate(csv.reader(f)):
        if idx == 0:
            continue  # skip the header row
        articles.append({"title": row[0], "text": row[1], "url": row[2], "source": row[3]})

print(f"{len(articles)} articles loaded")
print({k: (v[:60] + "…" if len(v) > 60 else v) for k, v in articles[0].items()})

23 articles loaded
{'title': 'Qwen3.8-2.4T-A95B Model Card', 'text': '# Qwen3.8-2.4T-A95B\n\n## Qwen3.8 Highlights\n\nQwen3.8 features…', 'url': 'https://huggingface.co/Qwen/Qwen3.8-2.4T-A95B', 'source': 'model_card'}


## 4. What Actually Makes a Good Embedding Model?

Five criteria cover almost every real decision:

1. **Retrieval quality on *your* data.** Public benchmarks rank models on public datasets; your corpus is neither. Benchmarks shortlist, your eval decides (Section 8).
2. **Price × dimensions.** You pay to *create* embeddings (per token) and forever after to *store and search* them (per dimension). A 3072-dim model doubles your vector-store bill versus a 1536-dim one for the same chunk count.
3. **Latency and batching.** Query embedding sits on the critical path of every user request; API round-trips versus local inference matters at scale.
4. **License and locality.** If your documents cannot leave your infrastructure, API embedders are disqualified no matter their score — this alone justifies the open-weight row below.
5. **Asymmetry support.** Good retrieval embedders encode *documents* and *queries* differently. This is the recurring production pattern, and every contender below has its own dialect for it.

**The leaderboard, with a health warning.** The [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) is the standard shortlisting tool. A snapshot **as of July 2026** (multilingual, retrieval-relevant models):

| Model | Access | Dims | Notes |
|---|---|---|---|
| Qwen3-Embedding-8B | open weights (Apache-2.0) | up to 4096 | top open model; the 0.6B sibling used here runs on a laptop |
| gemini-embedding-001 | API (course default) | 128–3072 (we use 1536) | top-tier API model |
| Cohere embed-v4.0 | API | 256–1536 | multimodal (text + images), strong retrieval focus |
| text-embedding-3-large | API | 256–3072 | OpenAI's larger option (3-small is our budget default) |

Treat scores as *indicative*: leaderboards reward benchmark fit, models can overfit them, and rankings shuffle every few months — which is precisely why this notebook ends with an eval you can rerun on your own data.

## 5. Asymmetry, in Three Dialects

Before indexing anything, look at the one idea all three contenders implement differently. Embed a document and a query with each model and read what the helper did:

- **`embed()`** sets Gemini's `task_type` (`RETRIEVAL_DOCUMENT` / `RETRIEVAL_QUERY`) or passes through for OpenAI.
- **`embed_cohere()`** sets `input_type` (`search_document` / `search_query`) — Cohere's API *requires* it.
- **`embed_local()`** applies the e5 family's `"query: "` / `"passage: "` prefixes, and a query *instruction* for instruction-tuned retrievers like Qwen3 (`query_prompt=` overrides it for anything it doesn't recognise).

Skipping the asymmetry does not raise an error. It quietly costs you accuracy — the most expensive kind of bug, and the reason all three helpers handle it for you rather than leaving it as a footnote.

`embed_cohere` also pins `output_dimension=1536` by default. `embed-v4.0` is Matryoshka-trained (256/512/1024/1536), so the dimension is a *choice*, and the course standardises on 1536 to keep vectors comparable and storage cost predictable.

In [6]:
sample_doc = "Qwen3.8 has 2.4T parameters in total, with 95B activated per token."
sample_query = "How big is the Qwen3.8 model?"

for name, fn in [("course embed()", embed), ("cohere embed-v4.0", embed_cohere)]:
    d = fn(sample_doc, task="document")[0]
    q = fn(sample_query, task="query")[0]
    print(f"{name:22} → {len(d)} dims (document) / {len(q)} dims (query)")

# Both API contenders land at 1536 here BY CHOICE (ours via EMBED_DIM, Cohere via
# output_dimension) — the open contender later lands at 1024, its native size.
# Criterion 2 (price × dimensions) made concrete — and vectors from different
# models are incomparable regardless, which is why each contender gets its own
# collection.

course embed()         → 1536 dims (document) / 1536 dims (query)
cohere embed-v4.0      → 1536 dims (document) / 1536 dims (query)


## 6. Contender 1 & 2: Two API Embedders, Two Indexes

`ingest()` and `search()` both accept an `embed_fn=` — any `(texts, task=...) -> vectors` callable. That single seam is how one pipeline hosts three embedders: same documents, same chunking, same ids, different vectors in different collections.

*(Vectors from different models live in unrelated spaces — never mix them in one collection. Separate collections is not tidiness, it is correctness.)*

`get_collection()` takes the same `embed_fn=` — not to embed anything, but to *check*. A Chroma collection locks its vector width on the first write and keeps it forever, so an index left over from an earlier run at a different dimension rejects today's vectors outright:

```
InvalidArgumentError: Collection expecting embedding with dimension of 3072, got 1536
```

The guard compares the locked width against what your embedder actually produces and rebuilds the collection when they disagree, printing a `↻` line so the re-ingest is never silent. Seeing that line once on a stale database is expected; seeing it on every run means something upstream keeps changing your dimension. To wipe an index deliberately, pass `reset=True`.

In [7]:
DB_PATH = "./embedding-bakeoff-chroma"

# embed_fn= is passed to BOTH calls for a reason: get_collection() uses it to
# check the index's locked vector width, ingest() uses it to fill the index.
col_course = get_collection("bakeoff_course", path=DB_PATH, embed_fn=embed)
ingest(articles, col_course, chunk_size=512, chunk_overlap=128, embed_fn=embed)
print("course default:", col_course.count(), "chunks")

course default: 233 chunks


In [8]:
col_cohere = get_collection("bakeoff_cohere", path=DB_PATH, embed_fn=embed_cohere)
ingest(articles, col_cohere, chunk_size=512, chunk_overlap=128, embed_fn=embed_cohere)
print("cohere embed-v4.0:", col_cohere.count(), "chunks")

cohere embed-v4.0: 233 chunks


**What just happened?** Two indexes, built by the same twenty-line ingestion loop, differing only in the function that turned text into numbers. `embed_cohere` set `input_type="search_document"` for you at index time and will set `search_query` at search time — the asymmetry handled once, in a function you can read a few cells up, rather than buried in a framework's node pipeline.

## 7. Contender 3: Open Weights on Your Machine

When documents cannot leave your infrastructure — or you want zero per-token costs — you run the embedder yourself. `embed_local()` wraps `sentence-transformers`; `Qwen3-Embedding-0.6B` (Apache-2.0, 1024-dim, ~600M parameters) is a strong small model from the family that tops the open-weight leaderboard as of July 2026.

Asymmetry returns in its third dialect. Instruction-tuned embedders like Qwen3 expect queries to carry an **instruction** while documents are encoded plain — and it is literally a text prefix:

```
Instruct: Given a web search query, retrieve relevant passages that answer the query
Query: <your question>
```

`embed_local()` knows this for the instruction-tuned families in its `_QUERY_PROMPTS` map (Qwen3, gte-Qwen) and applies it on `task="query"` only; the e5 family instead gets its `"query: "` / `"passage: "` prefixes on both sides. For a model the map doesn't recognise, pass `query_prompt="..."` yourself — the one thing you must not do is silently skip it.

The wrapper below exists only to bind the model name, so the function keeps the same `(texts, task=...)` contract every other embedder in this notebook uses.

*With the 0.6B default the first run downloads ~1.2 GB of weights; CPU is fine for a corpus this size (a GPU runtime is just faster). The 8B option is a different order of magnitude and wants a GPU.*

The dropdown also offers **`Qwen/Qwen3-Embedding-8B`**, the top-of-family sibling: same Apache-2.0 license and same instruction-aware query dialect, but up to 4096 dimensions (user-defined anywhere from 32 to 4096), a 32k context, and coverage of over 100 languages. It is the one to reach for when retrieval quality on your own data matters more than convenience, and when the documents genuinely cannot leave your infrastructure. The trade is hardware: eight billion parameters is roughly 16 GB of memory at half precision, so it wants a real GPU rather than a free Colab CPU runtime. **The bake-off below was run with the 0.6B**, which is why the 0.6B stays the default — selecting the 8B changes the numbers you will see.

In [9]:
def embed_open(texts, task="document"):
    """embed()-shaped view of the local model — embed_local applies the query instruction."""
    return embed_local(texts, model_name=OPEN_EMBED_MODEL, task=task)


col_open = get_collection("bakeoff_open", path=DB_PATH, embed_fn=embed_open)
ingest(articles, col_open, chunk_size=512, chunk_overlap=128, embed_fn=embed_open)
print(f"{OPEN_EMBED_MODEL}:", col_open.count(), "chunks")
print("dims:", len(embed_open(sample_doc)[0]))

# Proof the asymmetry is real: the SAME text embeds differently as a document and
# as a query, because only one of them carried the instruction.
probe = "How many parameters does Qwen3.8 have?"
print("same text, two roles → identical vectors?",
      embed_open(probe, task="document") == embed_open(probe, task="query"))

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen/Qwen3-Embedding-0.6B: 233 chunks
dims: 1024
same text, two roles → identical vectors? False


**What just happened?** A model running on this machine now speaks the same `(texts, task=...)` contract as two hosted APIs, so the rest of the notebook cannot tell them apart — and it got its query instruction without you remembering to add one. That contract is the entire portability story: keep the seam stable and the embedder becomes a swappable part. (Pick `bge-small` in the dropdown and no instruction is applied — it is not instruction-tuned, so the map correctly yields nothing; pick an e5 model and the `"query: "`/`"passage: "` prefix branch handles it instead.)

## 8. Three Indexes, One `search()`, One Eval

Now the honest comparison. First eyeball the same question through all three indexes, then measure with the course's own metrics — 📎 *hit rate and MRR, built in the RAG-evaluation notebook* — on questions generated from *this* corpus.

In [10]:
EMBEDDERS = {
    f"course ({EMBED_MODEL})": (col_course, embed),
    "cohere embed-v4.0": (col_cohere, embed_cohere),
    f"{OPEN_EMBED_MODEL} (local)": (col_open, embed_open),
}

question = "How many parameters does the Qwen3.8 model have?"

for name, (col, fn) in EMBEDDERS.items():
    print(f"◆ {name}")
    show_chunks(search(question, col, top_k=3, embed_fn=fn), max_chars=120)

◆ course (gemini-embedding-001)
  0.7828 | Qwen3.8-2.4T-A95B Model Card             | # Qwen3.8-2.4T-A95B  ## Qwen3.8 Highlights  Qwen3.8 features the following enhancements:  - Core Capabilities: Comprehen…
  0.7407 | Qwen3.8-2.4T-A95B Model Card             | 144 natively and extensible up to 1,010,000 tokens.  ## Benchmark Results  | | Opus 4.8 | Fable 5 | GPT 5.6 Sol (max) | …
  0.7299 | Qwen3.8-2.4T-A95B Model Card             |  Usage  Qwen3.8 comes with official support for `reasoning_effort`, which can be used to adjust reasoning depth and cont…
◆ cohere embed-v4.0
  0.5384 | Qwen3.8-2.4T-A95B Model Card             | # Qwen3.8-2.4T-A95B  ## Qwen3.8 Highlights  Qwen3.8 features the following enhancements:  - Core Capabilities: Comprehen…
  0.4695 | Qwen3.8-2.4T-A95B Model Card             |  Usage  Qwen3.8 comes with official support for `reasoning_effort`, which can be used to adjust reasoning depth and cont…
  0.4402 | Qwen3.8-2.4T-A95B Model Card             | 92.6 | 94.1 | 

**What just happened?** Three different models mostly agree on *which* chunks answer the question — competent embedders converge on easy queries — but the scores are **not comparable across models**. Each model has its own score distribution; 0.62 from one is not "worse" than 0.80 from another. Rankings are comparable; raw scores are not. That is exactly why we need per-question metrics rather than score-gazing.

📎 *`make_qa_pairs()` generates the eval set once from the shared chunks and saves it, so a rerun costs nothing. Because all three indexes carry identical chunk ids, one dataset grades all three.*

A cached eval file is only worth reusing if it grades *this* index, so `usable_eval_set()` checks it before we trust it: the right shape (a list of `{question, chunk_id}`), and chunk ids that still exist in the collection. Skip that check and a leftover file from another lesson raises `TypeError: string indices must be integers` in the next cell — or, far worse, grades cleanly against ids that no longer exist and reports a confident `hit_rate` of 0.00.

In [11]:
import json
from pathlib import Path

# 📎 make_qa_pairs() from "Evaluating Your RAG Pipeline" — generated once from
#    the SHARED chunk ids, saved, and reused by all three contenders. Two small
#    hardenings for the bake-off: the reply is normalised (whichever PROVIDER
#    you picked writes this eval set, and they don't all answer in one shape),
#    and the cached file is checked before it is trusted.
QA_GEN_PROMPT = """Context information is below.
---------------------
{context}
---------------------
Given ONLY the context above and no prior knowledge, write {n} quiz question(s)
that this context can answer on its own. Questions must be self-contained.

Return ONLY a JSON array of question strings, e.g. ["What is X?"]"""


def _parse_json(raw):
    return json.loads(raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip())


def _as_questions(parsed):
    """Coerce a model's reply into a flat list of question strings.

    Asked for ["What is X?"], models variously return the bare list,
    [{"question": "..."}], or {"questions": [...]}. Accept all three; drop
    anything else rather than letting it become a malformed pair.
    """
    if isinstance(parsed, dict):
        parsed = next((v for v in parsed.values() if isinstance(v, list)), [])
    questions = []
    for item in parsed if isinstance(parsed, list) else []:
        if isinstance(item, dict):
            item = item.get("question") or next((v for v in item.values() if isinstance(v, str)), "")
        if isinstance(item, str) and item.strip():
            questions.append(item.strip())
    return questions


def make_qa_pairs(collection, n_chunks=25, questions_per_chunk=1):
    """Generate (question, gold chunk_id) pairs from a sample of stored chunks."""
    sample = collection.get(limit=n_chunks, include=["documents"])
    qa_pairs = []
    for chunk_id, text in zip(sample["ids"], sample["documents"]):
        raw = generate(QA_GEN_PROMPT.format(context=text, n=questions_per_chunk))
        try:
            questions = _as_questions(_parse_json(raw))
        except json.JSONDecodeError:
            continue
        for q in questions[:questions_per_chunk]:
            qa_pairs.append({"question": q, "chunk_id": chunk_id})
    return qa_pairs


def usable_eval_set(pairs, collection):
    """Does this cached file actually grade THIS index?

    A saved eval set goes stale in two ways, and neither announces itself:
    a file from another lesson has a different shape (a TypeError one cell
    down), and a file from a differently-chunked run carries chunk_ids that no
    longer exist here — which is worse, because every hit_rate quietly reads
    0.00 and the table looks like a real result.
    """
    if not isinstance(pairs, list) or not pairs:
        return False
    if not all(isinstance(p, dict) and {"question", "chunk_id"} <= set(p) for p in pairs):
        return False
    known = set(collection.get(include=[])["ids"])  # ids only, no vectors fetched
    matched = sum(p["chunk_id"] in known for p in pairs)
    return matched > len(pairs) / 2  # a few stragglers are fine, a wholesale mismatch is not


EVAL_PATH = Path("embedding_eval_dataset.json")

cached = None
if EVAL_PATH.exists():
    try:
        cached = json.loads(EVAL_PATH.read_text())
    except json.JSONDecodeError:
        print(f"⚠️  {EVAL_PATH} is not valid JSON — regenerating")

if cached is not None and usable_eval_set(cached, col_course):
    qa_pairs = cached
    print(f"Loaded {len(qa_pairs)} question–chunk pairs from {EVAL_PATH}")
else:
    if cached is not None:
        print(f"⚠️  {EVAL_PATH} doesn't match this index (another lesson's file, or the "
              f"collection was rebuilt) — regenerating")
    qa_pairs = make_qa_pairs(col_course, n_chunks=20, questions_per_chunk=1)
    EVAL_PATH.write_text(json.dumps(qa_pairs, indent=2))
    print(f"Generated and saved {len(qa_pairs)} question–chunk pairs to {EVAL_PATH}")

print("each pair looks like:", qa_pairs[0])

Generated and saved 20 question–chunk pairs to embedding_eval_dataset.json
each pair looks like: {'question': 'According to the Qwen3.8-2.4T-A95B model overview, how many total parameters does the model have, and how many are activated?', 'chunk_id': 'Qwen3.8-2.4T-A95B Model Card-0'}


In [12]:
import pandas as pd


# 📎 Hit rate and MRR from "Evaluating Your RAG Pipeline", folded into one
#    evaluator that takes ANY (query, top_k) -> hits callable.
def evaluate_retrieval(qa_pairs, search_fn, top_k=5):
    """Hit rate + MRR for one retriever over the eval set."""
    hits = 0
    rr_total = 0.0
    for pair in qa_pairs:
        ids = [r["id"] for r in search_fn(pair["question"], top_k)]
        if pair["chunk_id"] in ids:
            hits += 1
            rr_total += 1 / (ids.index(pair["chunk_id"]) + 1)
    n = len(qa_pairs)
    return {"hit_rate": hits / n, "mrr": rr_total / n, "queries": n, "top_k": top_k}


def retriever_for(col, fn):
    """A (query, top_k) -> hits view of one index, for the evaluator."""
    return lambda query, top_k: search(query, col, top_k=top_k, embed_fn=fn)


reports = {
    name: evaluate_retrieval(qa_pairs, search_fn=retriever_for(col, fn), top_k=5)
    for name, (col, fn) in EMBEDDERS.items()
}

# Criterion 2 belongs in the same table as criterion 1 — you are choosing on both.
dims = {name: len(fn(sample_doc, task="document")[0]) for name, (_, fn) in EMBEDDERS.items()}

pd.DataFrame([{"model": name, **rep, "dims": dims[name]} for name, rep in reports.items()])

,model,hit_rate,mrr,queries,top_k,dims
0,course (gemini-embedding-001),0.85,0.633333,20,5,1536
1,cohere embed-v4.0,0.90,0.716667,20,5,1536
2,Qwen/Qwen3-Embedding-0.6B (local),0.85,0.660000,20,5,1024


**What just happened?** One table, three embedders, your corpus, with the storage dimension sitting next to the quality numbers — the decision the leaderboard could only shortlist (**your numbers will differ**; the eval questions are freshly generated on the first run). Two honest caveats before you crown a winner: with only ~20 questions on a small corpus, gaps of a few points are **noise, not signal** — rerun with more questions before switching models on this evidence; and overlap-sharing chunks can make near-misses count as misses. To see *which* questions each embedder lost, collect the missed pairs inside `evaluate_retrieval` (a five-line edit) — the misses are usually more informative than the average.

The method is the takeaway: identical chunks, identical questions, per-model rankings, and metrics you can rerun in minutes whenever a new model ships.

## 9. Swapping the Pipeline's Embedder

The punchline of the `embed()` seam: for the API providers, switching embedders is a **setup-cell edit** — change `EMBED_PROVIDER`/`EMBED_MODEL`, rerun, and nothing downstream changes, because the `EMBED_PROVIDER`/`EMBED_MODELS` globals in the helper cell are the only place the choice lives.

In [13]:
# The globals embed() reads are the single place the choice lives:
original = (EMBED_PROVIDER, EMBED_MODELS[EMBED_PROVIDER])
print("before:  ", EMBED_MODELS[EMBED_PROVIDER])

EMBED_PROVIDER = "openai"                        # the swap is two assignments…
EMBED_MODELS["openai"] = "text-embedding-3-small"
print("after:   ", EMBED_MODELS[EMBED_PROVIDER])

EMBED_PROVIDER = original[0]                     # …and undone the same way
EMBED_MODELS[EMBED_PROVIDER] = original[1]
print("restored:", EMBED_MODELS[EMBED_PROVIDER])

# (Every embed() call after an edit like this uses the new model — which is
# exactly why the swap must come with a full re-ingest; see the warning below.)

before:   gemini-embedding-001
after:    text-embedding-3-small
restored: gemini-embedding-001


One warning that saves real money: **switching embedders means re-embedding the entire corpus.** Vectors from different models (or the same model at different dimensions) live in unrelated spaces — never mix them in one index. Plan a full re-ingest, keep the old collection until the new one passes the eval above, and note that this is exactly why the three contenders got three collections rather than three runs against one.

For the record, the production AI Tutor's choice: **Gemini embeddings for dense retrieval**, fused with BM25 via RRF, then **Cohere reranking** on top — both vendors you just used directly, each where it is strongest. Reranking is where Part 1's advanced-RAG section picks this thread back up.

## 🔑 Key Takeaways

- Choose embedders on five axes — quality on *your* data, price × dimensions, latency, license/locality, and asymmetry support. The MTEB leaderboard shortlists; it does not decide.
- **Asymmetry is the recurring production pattern**: Gemini's `task_type`, Cohere's `input_type`, and open models' query instructions are one idea in three dialects — all three helpers apply it on `task="query"`, and `embed_local(query_prompt=...)` covers models the map doesn't know. Skipping it never errors; it just costs accuracy.
- Dimensions are a **choice**, not a property: `embed-v4.0` runs at 256–1536 and the course pins 1536 so vectors stay comparable and the storage bill stays predictable.
- The `embed_fn=` seam on `ingest()` and `search()` is what makes a bake-off possible: identical chunks, identical ids, identical eval, different vectors.
- Similarity **scores never compare across models**; rankings and metrics do. Hit rate + MRR on chunks with stable ids turn "which embedder?" into a table — just don't over-read tiny gaps on tiny samples.
- Switching embedders = re-embedding everything into a fresh collection. Budget for it, A/B it with the eval, and keep the `(texts, task=...)` contract stable so the rest of the code never notices.